# 10 - Pass 5 tuning: pair_weight and cleaned_drop_pct (dev only, GPU)

Two short dev-only runs before the main Pass 5 training (vault: Experiment Plan, "Pass 5 plan"). Same pattern as `w_tuning` / `fgm_tuning` in `02_train.ipynb`: small grid x 3 seeds, dev scored only, pick the winner, freeze it, then run the next step.

**What Pass 5 asks:** Pass 4's pairs cost 15-16 points of in-domain macro-F1 ([[Pass 4 Results]]). Does trusting the generated pairs less (`pair_weight` < 1) and/or removing the worst AI-tic pairs (`cleaned_drop_pct`) recover that loss?

**Run this notebook in two stages, because `cleaned_drop_tuning` needs `pair_weight_tuning`'s winner first:**
1. Set `PLAN = "pair_weight_tuning"` below, run through step 6, read the winning `pair_weight`.
2. Edit `src/gainsplit/plans.py` on GitHub: set `PASS5_PAIR_WEIGHT` to that value, commit and push.
3. Re-run this notebook (it re-clones the repo, so it picks up the new commit) with `PLAN = "cleaned_drop_tuning"`, read the winning `cleaned_drop_pct`.
4. Set `PASS5_CLEANED_DROP_PCT` too, commit and push. Both constants are now frozen for the main `pass5` plan.

**Before you run, in the right-hand panel:**
- **Input: the `gainsplit-processed` version that has `pairs_train.csv`, `manifest_pairs.json`, `pair_shortcut_scores.csv` and `manifest_shortcut_scores.json`** (the last two are needed for `cleaned_drop_tuning` only; add them first if you haven't).
- **Internet: on.**
- **Accelerator: GPU T4 x2.** 18 runs total (9 + 9), each a few minutes (10% labels, dev only, no masked tests, no explanations): about 15-25 minutes per stage. An interactive run is fine here (small, quick); no need to commit.

## 1. Get the code

In [ ]:
import os, subprocess, sys, time
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Tests (about a minute)
If anything fails, stop and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Check the GPUs and the data
Checks `gainsplit-processed` against `manifest.json`, then loads the `unfiltered` pair set (needed by both tuning plans: `pair_weight_tuning` trains on it directly; `cleaned_drop_tuning` derives `cleaned` from it) and, if present, the shortcut-score file `cleaned_drop_tuning` needs.

In [ ]:
import torch
from gainsplit.data import find_data_dir, verify, load_pairs, load_shortcut_scores

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")
_, info = load_pairs("unfiltered", DATA_DIR, check_against=DATA_DIR)
print("unfiltered pairs:", info["n_rows"], "rows,", info["n_seeds"], "seeds")
try:
    scores = load_shortcut_scores(DATA_DIR)
    print("shortcut scores: found,", len(scores), "seeds (needed for cleaned_drop_tuning)")
except FileNotFoundError as e:
    print("shortcut scores: not found -", e, "\n(fine for pair_weight_tuning; add the file before cleaned_drop_tuning)")

## 4. Choose the plan
- `pair_weight_tuning`: arm **A10** at **10%** labels, unfiltered pairs, `pair_weight` in **0.1, 0.3, 1.0**, seeds **0, 1, 2** = 9 runs, dev only.
- `cleaned_drop_tuning`: arm **A16** at **10%** labels, the **cleaned** pair set, `cleaned_drop_pct` in **10, 20, 30**, seeds **0, 1, 2** = 9 runs, dev only, at the **frozen `PASS5_PAIR_WEIGHT`** (set it in `plans.py`, push, re-clone, before running this).

In [ ]:
from gainsplit.plans import PLANS

PLAN = "cleaned_drop_tuning"   # stage 2; stage 1 ("pair_weight_tuning") done 2026-11-14, PASS5_PAIR_WEIGHT = 0.1
OUT_DIR = f"/kaggle/working/results/{PLAN}"
configs = PLANS[PLAN]()
for c in configs:
    print(f"{c.run_id:20s} arm={c.arm:3s} labels={c.budget:3d}% seed={c.seed} pair_weight={c.pair_weight:g} "
          f"cleaned_drop_pct={c.cleaned_drop_pct:g} test scored: {c.eval_test}")
print(len(configs), "runs")

## 5. Run
One run per GPU; if the session stops halfway, run this cell again (finished runs are skipped).

In [ ]:
from gainsplit.runner import run_all

table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
table

## 6. Read the result
Mean and std over the 3 seeds, per setting. `dev_macro_f1` is the column to read (both plans score dev only).

**`pair_weight_tuning`:** pick the `pair_weight` with the best mean dev macro-F1. Tie within about 1 std: prefer the **smaller** weight (same tie-break rule as every other `_tuning` plan) - a smaller weight trusts the pairs less, which is the direction Pass 5 is testing for. A10's dev macro-F1 at `pair_weight=1.0` should be noticeably below A1's normal 10% dev score (about 0.72-0.73, matching Pass 4's A10); if every weight scores about that low, something is off and down-weighting is not acting as expected - stop and report.

**`cleaned_drop_tuning`:** pick the `cleaned_drop_pct` with the best mean dev macro-F1. Tie: prefer the **smaller** drop_pct (keep more data). Compare against the `pair_weight_tuning` winner's score at the same weight (cleaned_drop_pct effectively 0): the cleaned runs should score at or above that baseline if cleaning helps at all.

In [ ]:
keys = ["arm", "budget", "pair_weight", "cleaned_drop_pct"]
cols = [c for c in ["dev_macro_f1", "dev_accuracy", "dev_roc_auc", "minutes"] if c in table.columns]
summary = table.groupby(keys)[cols].agg(["mean", "std"]).round(4)
display(summary)
print("runs per group:"); print(table.groupby(keys).size().to_string())

## 7. Download
Interactive session: download `results_<plan>.zip` before closing (Kaggle deletes `/kaggle/working` on an interactive session's end, unlike a commit).

In [ ]:
import shutil
print(shutil.make_archive(f"/kaggle/working/results_{PLAN}", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")